# ĐỌC ĐA ĐỊNH DẠNG FILE EXCEL/CSV VÀ TỰ ĐỘNG DÒ CỘT PHẢN HỒI

In [1]:
import os
import io
import re
import csv
import json
import random
import unicodedata
import joblib
import pandas as pd
import numpy as np

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
MESSY_DIR = os.path.join(ROOT, 'data', 'samples_messy')
MODEL_DIR = os.path.join(ROOT, 'models')
os.makedirs(MESSY_DIR, exist_ok=True)

## 1. TẠO 10 FILE EXCEL/CSV 'BẨN' GIẢ LẬP CÁC TRƯỜNG ĐẠI HỌC

In [3]:
#Lấy các câu phản hồi thật từ UIT-VSFC trộn thêm nhiễu (mã SV, SĐT, câu trống, nhiều câu) để tạo 10 file khảo sát khác nhau
rng = random.Random(42)
sents_pool = [x.strip() for x in open(os.path.join(RAW_DIR, 'train_sentences.txt'), encoding='utf-8').read().splitlines() if len(x.strip()) > 20]
rng.shuffle(sents_pool)
len(sents_pool)

10500

In [4]:
courses = ['Cấu trúc dữ liệu', 'Giải tích 1', 'Lập trình Web', 'Kinh tế vi mô', 'Cơ sở dữ liệu']
lecturers = ['Nguyễn Văn An', 'Trần Thị Bình', 'Lê Hoàng Cường', 'Phạm Minh Đức']
noise_vals = ['', 'không', 'ko có ý kiến', 'k', 'Không có', 'ok', '-', 'N/A', '  ']

idx_ptr = 0
def get_chunk(n):
    global idx_ptr
    chunk = sents_pool[idx_ptr : idx_ptr + n]
    idx_ptr += n
    out = []
    for k, text in enumerate(chunk):
        if k % 7 == 0:
            out.append(rng.choice(noise_vals))
        elif k % 11 == 0:
            out.append('Thầy ' + rng.choice(lecturers) + ' dạy ' + text + ' mssv 21520199 sđt 0912345678')
        else:
            out.append(text)
    return out

In [5]:
#Tạo 10 file với đủ các bẫy: dòng tiêu đề ở row 5, tiếng Anh, nhiều sheet, .xls cũ, csv chấm phẩy, cp1258, 2 cột chữ, ô gộp, HTML giả .xls
truth = {}

#File 1: chuẩn sạch
n = 100
pd.DataFrame({'STT': range(1, n+1), 'MSSV': [21520000+k for k in range(n)], 'Họ tên': ['SV '+str(k) for k in range(n)], 'Ý kiến đóng góp': get_chunk(n)}).to_excel(os.path.join(MESSY_DIR, '01_clean.xlsx'), index=False)
truth['01_clean.xlsx'] = {'header_row': 0, 'text_cols': ['Ý kiến đóng góp']}

#File 2: có 5 dòng tiêu đề trường ở đầu, header thật ở dòng 5, kèm điểm Likert 1-5
n = 120
body2 = pd.DataFrame({'STT': range(1, n+1), 'Mã SV': [21520000+k for k in range(n)], 'Điểm giảng dạy (1-5)': [rng.randint(1, 5) for _ in range(n)], 'Mức độ hài lòng': [rng.randint(1, 5) for _ in range(n)], 'Góp ý': get_chunk(n)})
with pd.ExcelWriter(os.path.join(MESSY_DIR, '02_title_rows.xlsx'), engine='openpyxl') as w:
    pd.DataFrame([['TRƯỜNG ĐẠI HỌC ABC'], ['PHÒNG KHẢO THÍ & ĐBCL'], ['KẾT QUẢ KHẢO SÁT MÔN HỌC HK1'], ['Môn: Cấu trúc dữ liệu'], ['']]).to_excel(w, index=False, header=False, startrow=0)
    body2.to_excel(w, index=False, startrow=5)
truth['02_title_rows.xlsx'] = {'header_row': 5, 'text_cols': ['Góp ý']}

#File 3: tên cột tiếng Anh
n = 90
pd.DataFrame({'ID': [21520000+k for k in range(n)], 'Course': [rng.choice(courses) for _ in range(n)], 'Student_Feedback': get_chunk(n), 'Rating': [rng.randint(1, 5) for _ in range(n)]}).to_excel(os.path.join(MESSY_DIR, '03_english_cols.xlsx'), index=False)
truth['03_english_cols.xlsx'] = {'header_row': 0, 'text_cols': ['Student_Feedback']}

#File 4: nhiều sheet (bìa, tổng hợp, dữ liệu)
n = 110
with pd.ExcelWriter(os.path.join(MESSY_DIR, '04_multi_sheet.xlsx'), engine='openpyxl') as w:
    pd.DataFrame({'Hướng dẫn': ['Sheet DuLieu chứa phản hồi.', 'Không sửa cột.']}).to_excel(w, sheet_name='HuongDan', index=False)
    pd.DataFrame({'Môn': courses, 'Sĩ số': [50, 60, 45, 70, 55]}).to_excel(w, sheet_name='TongHop', index=False)
    pd.DataFrame({'Mã môn': ['CS101']*n, 'Nhận xét của sinh viên': get_chunk(n)}).to_excel(w, sheet_name='DuLieu', index=False)
truth['04_multi_sheet.xlsx'] = {'header_row': 0, 'text_cols': ['Nhận xét của sinh viên']}

#File 5: định dạng .xls cũ (Excel 97-2003) tạo bằng xlwt
n = 80
import xlwt
wb5 = xlwt.Workbook()
ws5 = wb5.add_sheet('Sheet1')
for c_idx, h_val in enumerate(['STT', 'Ý kiến khác', 'Học kỳ']):
    ws5.write(0, c_idx, h_val)
for r_idx, txt_val in enumerate(get_chunk(n), start=1):
    ws5.write(r_idx, 0, r_idx)
    ws5.write(r_idx, 1, txt_val)
    ws5.write(r_idx, 2, 'HK1')
wb5.save(os.path.join(MESSY_DIR, '05_legacy.xls'))
truth['05_legacy.xls'] = {'header_row': 0, 'text_cols': ['Ý kiến khác']}

In [6]:
#File 6: CSV UTF-8-SIG dùng dấu chấm phẩy ;
n = 100
pd.DataFrame({'Mã SV': [21520000+k for k in range(n)], 'Giảng viên': [rng.choice(lecturers) for _ in range(n)], 'Góp ý': get_chunk(n)}).to_csv(os.path.join(MESSY_DIR, '06_semicolon_bom.csv'), index=False, sep=';', encoding='utf-8-sig')
truth['06_semicolon_bom.csv'] = {'header_row': 0, 'text_cols': ['Góp ý']}

#File 7: CSV bảng mã UTF-16 (Excel 'Unicode Text' rất hay gặp ở các trường)
n = 80
pd.DataFrame({'STT': range(1, n+1), 'Phản hồi': get_chunk(n)}).to_csv(os.path.join(MESSY_DIR, '07_utf16_tab.csv'), index=False, sep='\t', encoding='utf-16')
truth['07_utf16_tab.csv'] = {'header_row': 0, 'text_cols': ['Phản hồi']}

#File 8: khảo sát có 2 cột văn bản riêng (Ưu điểm & Nhược điểm)
n = 90
pd.DataFrame({'MSSV': [21520000+k for k in range(n)], 'Ưu điểm': get_chunk(n), 'Nhược điểm': get_chunk(n), 'Điểm GV': [rng.randint(1, 5) for _ in range(n)]}).to_excel(os.path.join(MESSY_DIR, '08_two_text_cols.xlsx'), index=False)
truth['08_two_text_cols.xlsx'] = {'header_row': 0, 'text_cols': ['Ưu điểm', 'Nhược điểm']}

#File 9: có ô gộp (Merged Cells), đoạn văn dài nhiều câu và dòng lặp tiêu đề giữa file
n = 60
long_p = [sents_pool[k] + '. ' + sents_pool[k+1] for k in range(0, 2*n, 2)]
d9 = pd.DataFrame({'STT': range(1, n+1), 'Họ và tên': ['Nguyễn Văn A']*n, 'Ý kiến góp ý': long_p})
d9.to_excel(os.path.join(MESSY_DIR, '09_long_paragraphs.xlsx'), index=False)
import openpyxl
wb = openpyxl.load_workbook(os.path.join(MESSY_DIR, '09_long_paragraphs.xlsx'))
ws = wb.active
ws.insert_rows(1, 2)
ws['A1'] = 'BÁO CÁO KHẢO SÁT HỌC PHẦN'
ws.merge_cells('A1:C1')
ws.insert_rows(30)
for col_i, val in enumerate(['STT', 'Họ và tên', 'Ý kiến góp ý'], start=1):
    ws.cell(row=30, column=col_i, value=val)
wb.save(os.path.join(MESSY_DIR, '09_long_paragraphs.xlsx'))
truth['09_long_paragraphs.xlsx'] = {'header_row': 2, 'text_cols': ['Ý kiến góp ý']}

#File 10: file mang đuôi .xls nhưng ruột là bảng HTML
n = 70
pd.DataFrame({'Mã': [21520000+k for k in range(n)], 'Lớp': ['CNTT01']*n, 'Comment': get_chunk(n)}).to_html(os.path.join(MESSY_DIR, '10_html_as_xls.xls'), index=False)
truth['10_html_as_xls.xls'] = {'header_row': 0, 'text_cols': ['Comment']}

with open(os.path.join(MESSY_DIR, 'ground_truth.json'), 'w', encoding='utf-8') as f:
    json.dump(truth, f, ensure_ascii=False, indent=2)

In [7]:
sorted(os.listdir(MESSY_DIR))

['01_clean.xlsx',
 '02_title_rows.xlsx',
 '03_english_cols.xlsx',
 '04_multi_sheet.xlsx',
 '05_legacy.xls',
 '06_semicolon_bom.csv',
 '07_utf16_tab.csv',
 '08_two_text_cols.xlsx',
 '09_long_paragraphs.xlsx',
 '10_html_as_xls.xls',
 'ground_truth.json']

## 2. HÀM ĐỌC ĐA ĐỊNH DẠNG (SNIFF MAGIC BYTES & ENCODING)

In [8]:
#Kiểm tra 8 byte đầu tiên của file 10_html_as_xls.xls xem tại sao pd.read_excel thường bị lỗi
tempt_bytes = open(os.path.join(MESSY_DIR, '10_html_as_xls.xls'), 'rb').read(60)
tempt_bytes

b'<table border="1" class="dataframe">\n  <thead>\n    <tr style'

In [9]:
#Nhận diện định dạng thật bằng magic bytes thay vì tin vào đuôi file
import chardet

def read_any_file(filepath):
    data = open(filepath, 'rb').read()
    head = data[:2048].lstrip().lower()
    if data[:4] == b'PK\x03\x04':
        return pd.read_excel(io.BytesIO(data), sheet_name=None, header=None, dtype=object, engine='openpyxl')
    if data[:8] == b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1':
        return pd.read_excel(io.BytesIO(data), sheet_name=None, header=None, dtype=object, engine='xlrd')
    if head.startswith(b'<html') or head.startswith(b'<table') or head.startswith(b'<!doctype'):
        text = data.decode('utf-8', errors='replace')
        tabs = pd.read_html(io.StringIO(text))
        out_tabs = {}
        for i, t in enumerate(tabs):
            hdr_df = pd.DataFrame([list(t.columns)], columns=range(t.shape[1]))
            body_df = pd.DataFrame(t.values, columns=range(t.shape[1]))
            out_tabs[f'table_{i+1}'] = pd.concat([hdr_df, body_df], ignore_index=True).astype(object)
        return out_tabs
    #Còn lại là CSV/TSV: tự dò bảng mã (kể cả UTF-16 có BOM) và dấu phân cách
    enc_guess = chardet.detect(data[:100000]).get('encoding') or 'utf-8'
    for enc in ['utf-8-sig', 'utf-16', enc_guess, 'utf-8', 'cp1258', 'latin-1']:
        try:
            text = data.decode(enc)
            break
        except Exception:
            continue
    try:
        sep = csv.Sniffer().sniff(text[:10000], delimiters=',;\t|').delimiter
    except Exception:
        sep = ','
    df = pd.read_csv(io.StringIO(text), sep=sep, header=None, dtype=object, engine='python', on_bad_lines='skip', skip_blank_lines=False)
    return {'csv': df}

In [10]:
#Thử đọc file 04_multi_sheet.xlsx và chọn sheet có nhiều ô văn bản dài nhất
sheets_04 = read_any_file(os.path.join(MESSY_DIR, '04_multi_sheet.xlsx'))
list(sheets_04.keys())

['HuongDan', 'TongHop', 'DuLieu']

In [11]:
def pick_best_sheet(sheets_dict):
    best_name, best_score = None, -1
    for name, df_s in sheets_dict.items():
        if df_s is None or df_s.dropna(how='all').empty:
            continue
        long_cells = df_s.map(lambda x: isinstance(x, str) and len(x.strip()) > 15).sum().sum()
        score = float(long_cells) + len(df_s) * 0.1
        if score > best_score:
            best_score, best_name = score, name
    return best_name

pick_best_sheet(sheets_04)

'DuLieu'

## 3. TỰ ĐỘNG DÒ DÒNG TIÊU ĐỀ (HEADER ROW) VÀ DỰNG BẢNG

In [12]:
#Xem 8 dòng đầu của file 02_title_rows.xlsx (có quốc hiệu và tên trường ở trên)
raw_02 = read_any_file(os.path.join(MESSY_DIR, '02_title_rows.xlsx'))['Sheet1']
raw_02.head(8)

,0,1,2,3,4
0,TRƯỜNG ĐẠI HỌC ABC,NaN,NaN,NaN,NaN
1,PHÒNG KHẢO THÍ & ĐBCL,NaN,NaN,NaN,NaN
2,KẾT QUẢ KHẢO SÁT MÔN HỌC HK1,NaN,NaN,NaN,NaN
3,Môn: Cấu trúc dữ liệu,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
5,STT,Mã SV,Điểm giảng dạy (1-5),Mức độ hài lòng,Góp ý
6,1,21520000,2,1,ok
7,2,21520001,3,1,"có kiến thức nền tảng này , sinh viên có thể h..."


In [13]:
def strip_accents(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn').replace('đ', 'd').replace('Đ', 'D')
    return re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip()

kw_text = ['y kien', 'gop y', 'nhan xet', 'phan hoi', 'danh gia', 'comment', 'feedback', 'review', 'noi dung', 'uu diem', 'nhuoc diem', 'y kien khac']
kw_meta = ['mssv', 'ma sv', 'id', 'stt', 'ho ten', 'mon hoc', 'course', 'giang vien', 'hoc ky', 'diem', 'rating', 'hai long']

def is_num(val):
    try:
        float(str(val).replace(',', '.'))
        return True
    except Exception:
        return False

In [14]:
def detect_header_row(raw_df, max_scan=30):
    n = min(max_scan, len(raw_df))
    n_cols = raw_df.shape[1]
    best_row, best_score = 0, -1.0
    for i in range(n):
        row = raw_df.iloc[i]
        vals = [str(v).strip() for v in row if pd.notna(v) and str(v).strip() != '']
        if len(vals) < max(2, 0.4 * n_cols):
            continue
        non_num = sum(not is_num(v) for v in vals) / len(vals)
        short_str = sum(len(v) <= 60 for v in vals) / len(vals)
        uniq = len(set(strip_accents(v) for v in vals)) / len(vals)
        fill = len(vals) / n_cols
        kw_hits = sum(any(k in strip_accents(v) for k in kw_text + kw_meta) for v in vals)
        score = non_num * 2.0 + short_str + uniq + fill * 2.0 + kw_hits * 0.8
        if score > best_score:
            best_score, best_row = score, i
    return best_row

detect_header_row(raw_02)

5

In [15]:
def build_clean_table(raw_df, header_row):
    hdr = [str(h).strip() if pd.notna(h) and str(h).strip() else f'col_{j+1}' for j, h in enumerate(raw_df.iloc[header_row])]
    df = raw_df.iloc[header_row + 1:].copy()
    df.columns = hdr
    df = df.dropna(how='all').dropna(axis=1, how='all').reset_index(drop=True)
    #Bỏ các dòng lặp lại tiêu đề ở giữa bảng (do xuất báo cáo chia trang)
    same_hdr = df.apply(lambda r: sum(str(a).strip() == str(b).strip() for a, b in zip(r, df.columns)) >= max(2, len(df.columns)//2), axis=1)
    return df[~same_hdr].reset_index(drop=True)

tab_02 = build_clean_table(raw_02, detect_header_row(raw_02))
tab_02.head(3)

,STT,Mã SV,Điểm giảng dạy (1-5),Mức độ hài lòng,Góp ý
0,1,21520000,2,1,ok
1,2,21520001,3,1,"có kiến thức nền tảng này , sinh viên có thể h..."
2,3,21520002,1,2,cho bài tập nhiều quá không làm hết được .


## 4. CHẤM ĐIỂM TỰ ĐỘNG DÒ CỘT PHẢN HỒI (COLUMN DETECTION)

In [16]:
vn_chars = re.compile(r'[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]', re.I)

def detect_text_columns(df):
    scores = []
    for col in df.columns:
        s = df[col].dropna().astype(str).str.strip()
        s = s[s != '']
        if len(s) == 0:
            continue
        h_norm = strip_accents(col)
        name_score = 1.0 if any(k == h_norm for k in kw_text) else (0.85 if any(k in h_norm for k in kw_text) else 0.0)
        num_mask = s.map(is_num).astype(bool)
        num_ratio = float(num_mask.mean())
        text_s = s[~num_mask]
        if len(text_s) > 0:
            avg_len = float(text_s.str.len().mean())
            multi_word = float((text_s.str.split().str.len() >= 3).sum()) / len(s)
            vn_ratio = float(text_s.str.contains(vn_chars).sum()) / len(s)
        else:
            avg_len, multi_word, vn_ratio = 0.0, 0.0, 0.0
        uniq_ratio = float(s.nunique()) / len(s)
        content_score = 0.35 * min(avg_len / 50.0, 1.0) + 0.30 * multi_word + 0.20 * vn_ratio + 0.15 * min(uniq_ratio * 1.5, 1.0)
        total = 0.45 * name_score + 0.55 * content_score
        if num_ratio > 0.5:
            total *= 0.1
        if s.nunique() <= 6 and len(s) > 25:
            total *= 0.2
        scores.append((col, round(float(total), 3), round(float(avg_len), 1)))
    scores.sort(key=lambda x: -x[1])
    top_score = scores[0][1] if scores else 0
    picked = [c for c, sc, _ in scores if sc >= max(0.35, 0.72 * top_score)]
    return picked, scores

In [17]:
detect_text_columns(tab_02)

(['Góp ý'],
 [('Góp ý', 0.976, 61.0),
  ('STT', 0.008, 0.0),
  ('Mã SV', 0.008, 0.0),
  ('Điểm giảng dạy (1-5)', 0.0, 0.0),
  ('Mức độ hài lòng', 0.0, 0.0)])

## 5. KIỂM ĐỊNH TRÊN CẢ 10 FILE EXCEL/CSV 'BẨN'

In [18]:
results_check = []
for fname, ans in sorted(truth.items()):
    fpath = os.path.join(MESSY_DIR, fname)
    sheets = read_any_file(fpath)
    sh = pick_best_sheet(sheets)
    hr = detect_header_row(sheets[sh])
    tab = build_clean_table(sheets[sh], hr)
    picked_cols, sc_list = detect_text_columns(tab)
    ok_hr = (hr == ans['header_row'])
    ok_cols = (set(picked_cols) == set(ans['text_cols']))
    results_check.append({
        'file': fname,
        'sheet': sh,
        'header_row': f"{hr} (expect {ans['header_row']})",
        'detected_text_cols': ', '.join(picked_cols),
        'n_rows': len(tab),
        'pass': ok_hr and ok_cols
    })

df_check = pd.DataFrame(results_check)
df_check

,file,sheet,header_row,detected_text_cols,n_rows,pass
0,01_clean.xlsx,Sheet1,0 (expect 0),Ý kiến đóng góp,100,True
1,02_title_rows.xlsx,Sheet1,5 (expect 5),Góp ý,120,True
2,03_english_cols.xlsx,Sheet1,0 (expect 0),Student_Feedback,90,True
3,04_multi_sheet.xlsx,DuLieu,0 (expect 0),Nhận xét của sinh viên,110,True
4,05_legacy.xls,Sheet1,0 (expect 0),Ý kiến khác,80,True
5,06_semicolon_bom.csv,csv,0 (expect 0),Góp ý,100,True
6,07_utf16_tab.csv,csv,0 (expect 0),Phản hồi,80,True
7,08_two_text_cols.xlsx,Sheet1,0 (expect 0),"Ưu điểm, Nhược điểm",90,True
8,09_long_paragraphs.xlsx,Sheet1,2 (expect 2),Ý kiến góp ý,60,True
9,10_html_as_xls.xls,table_1,0 (expect 0),Comment,70,True


In [19]:
print('Tỷ lệ dò đúng hoàn toàn trên 10 file bẩn:', df_check['pass'].sum(), '/', len(df_check))

Tỷ lệ dò đúng hoàn toàn trên 10 file bẩn: 10 / 10


## 6. CHẠY THỬ PIPELINE DỰ ĐOÁN VÀ XUẤT BÁO CÁO EXCEL MẪU CHO GIẢNG VIÊN

In [20]:
#Lấy file 02_title_rows.xlsx (có điểm Likert + dòng tiêu đề lệch), lọc câu rỗng, che PII, chạy model dự đoán và xuất file kết quả
tfidf = joblib.load(os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))
lr_sent = joblib.load(os.path.join(MODEL_DIR, 'lr_sentiment.pkl'))
lr_top = joblib.load(os.path.join(MODEL_DIR, 'lr_topic.pkl'))

In [21]:
sent_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
topic_map = {0: 'lecturer', 1: 'training_program', 2: 'facility', 3: 'others'}
no_content_set = {'', 'k', 'ko', 'khong', 'không', 'không có', 'ko có ý kiến', 'không có ý kiến', 'ok', '-', 'n/a'}

#Bỏ cột Mã SV để đảm bảo ẩn danh cho sinh viên
df_export = tab_02.drop(columns=['Mã SV'], errors='ignore').copy()
text_col = 'Góp ý'
df_export['text_clean'] = df_export[text_col].fillna('').astype(str).str.strip()
#Che thông tin cá nhân nếu sinh viên lỡ viết vào câu góp ý
df_export['text_clean'] = df_export['text_clean'].apply(lambda s: re.sub(r'(?<!\d)\d{7,10}(?!\d)', '[HIDDEN_ID]', s))
df_export['is_no_content'] = df_export['text_clean'].apply(lambda s: s.lower() in no_content_set or len(s) < 3)

In [22]:
valid_mask = ~df_export['is_no_content']
X_excel = tfidf.transform(df_export.loc[valid_mask, 'text_clean'].str.lower())

df_export['sentiment_pred'] = 'NO_CONTENT'
df_export['sentiment_conf'] = np.nan
df_export['topic_pred'] = 'NO_CONTENT'

s_probs = lr_sent.predict_proba(X_excel)
t_preds = lr_top.predict(X_excel)
df_export.loc[valid_mask, 'sentiment_pred'] = [sent_map[i] for i in s_probs.argmax(axis=1)]
df_export.loc[valid_mask, 'sentiment_conf'] = s_probs.max(axis=1).round(3)
df_export.loc[valid_mask, 'topic_pred'] = [topic_map[i] for i in t_preds]

In [23]:
#Gắn cờ needs_review nếu độ tự tin < 0.65 hoặc điểm hài lòng Likert mâu thuẫn với cảm xúc dự đoán
likert_num = pd.to_numeric(df_export['Mức độ hài lòng'], errors='coerce')
df_export['needs_review'] = (
    (df_export['sentiment_conf'] < 0.65)
    | ((likert_num <= 2) & (df_export['sentiment_pred'] == 'positive'))
    | ((likert_num >= 4) & (df_export['sentiment_pred'] == 'negative'))
)

In [24]:
df_export.head(10)

,STT,Điểm giảng dạy (1-5),Mức độ hài lòng,Góp ý,text_clean,is_no_content,sentiment_pred,sentiment_conf,topic_pred,needs_review
0,1,2,1,ok,ok,True,NO_CONTENT,NaN,NO_CONTENT,False
1,2,3,1,"có kiến thức nền tảng này , sinh viên có thể h...","có kiến thức nền tảng này , sinh viên có thể h...",False,negative,0.483,others,True
2,3,1,2,cho bài tập nhiều quá không làm hết được .,cho bài tập nhiều quá không làm hết được .,False,negative,0.849,training_program,False
3,4,5,3,các code ví dụ không được chiếu hết trên slide .,các code ví dụ không được chiếu hết trên slide .,False,negative,0.836,training_program,False
4,5,4,4,vì môn học này không hay .,vì môn học này không hay .,False,negative,0.815,training_program,True
5,6,2,3,"hệ thống slide bài giảng chi tiết , rõ ràng .","hệ thống slide bài giảng chi tiết , rõ ràng .",False,positive,0.699,training_program,False
6,7,1,2,tiết học chưa được thu hút .,tiết học chưa được thu hút .,False,negative,0.945,training_program,False
7,8,1,2,-,-,True,NO_CONTENT,NaN,NO_CONTENT,False
8,9,3,4,thầy cho khá nhiều bài tập đa dạng .,thầy cho khá nhiều bài tập đa dạng .,False,positive,0.714,lecturer,False
9,10,2,2,quá cứng nhắc trong quá trình cho điểm .,quá cứng nhắc trong quá trình cho điểm .,False,negative,0.840,lecturer,False


In [25]:
out_excel_path = os.path.join(PROC_DIR, 'sample_excel_predicted.xlsx')
df_export.to_excel(out_excel_path, index=False)
print('Đã xuất file Excel kết quả mẫu tại:', out_excel_path)

Đã xuất file Excel kết quả mẫu tại: d:\VSFC-Feedback-Analyzer\data\processed\sample_excel_predicted.xlsx
